# Панель исследования по конфигам

Этот notebook — **новая версия** поверх рабочего слоя.

Что важно:
- старые [synthetic_api.py](synthetic_api.py) и [synthetic_control_panel.ipynb](synthetic_control_panel.ipynb) не меняются по смыслу;
- здесь добавлен отдельный слой для **сохраняемых конфигов исследования**;
- можно сохранить базовый сценарий, набор методов, оси перебора и потом воспроизводить исследование из файла.


In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'synthetic_study_api.py').exists():
    PROJECT_DIR = Path('/Users/karimau/Projects_C++/Course Project')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import synthetic_api as api
import synthetic_study_api as study

PROJECT_DIR


In [ ]:
def show_table(obj, n=5):
    if hasattr(obj, 'head'):
        return obj.head(n)
    return obj[:n]


## 1. Базовый сценарий

Здесь задается один базовый сценарий. Дальше поверх него можно делать исследование: менять ранг, шум, тип пропусков и так далее.


In [ ]:
base_scenario = api.ScenarioConfig(
    matrix=api.MatrixConfig(
        m=60,
        n=60,
        rank=4,
        factor_distribution='gaussian',
        singular_value_profile='linear',
        singular_scale=1.0,
        coherence_mode='incoherent',
    ),
    structure=api.StructureConfig(mode='none'),
    missingness_field=api.MissingnessFieldConfig(mode='random'),
    mask_sampling=api.MaskSamplingConfig(observed_fraction=0.35, exact_fraction=True),
    split=api.SplitConfig(validation_fraction=0.15, test_fraction=0.15),
    noise=api.NoiseConfig(mode='gaussian', std=0.02),
)

base_scenario


## 2. Методы и оси перебора

Теперь методы и переборы описываются отдельно от сценария.


In [ ]:
method_specs = [
    study.MethodSpec(name='soft_impute', label='Soft-Impute', params={'rank': 4, 'max_iter': 80}),
    study.MethodSpec(name='rgd', label='RGD', params={'rank': 4, 'max_iter': 120, 'init': 'spectral'}),
    study.MethodSpec(name='rgd_l2', label='RGD + L2', params={'rank': 4, 'max_iter': 120, 'init': 'spectral', 'l2_reg': 0.03}),
]

sweeps = [
    study.SweepAxis(parameter_path='matrix.rank', values=[4, 6, 8, 10]),
]

method_specs, sweeps


## 3. Конфиг исследования

Это уже не просто один сценарий, а полноценное описание исследования.


In [ ]:
study_config = study.StudyConfig(
    name='rank_sweep_demo',
    description='Перебор истинного ранга при фиксированных остальных параметрах.',
    base_scenario=base_scenario,
    methods=method_specs,
    sweeps=sweeps,
    seeds=[41, 42],
    output_dir='study_outputs/rank_sweep_demo',
)

study_config


## 4. Предпросмотр сетки

Полезно сначала посмотреть, какие именно сценарии будут построены.


In [ ]:
preview_rows = study.preview_study_grid(study_config)
show_table(preview_rows)


## 5. Сохранение и загрузка конфига

Это и есть главный смысл новой версии: исследование можно сохранить в файл, а потом загрузить обратно.


In [ ]:
config_path = PROJECT_DIR / 'study_config_rank_sweep_demo.json'
study.save_study_config(study_config, config_path)
loaded_config = study.load_study_config(config_path)
config_path, loaded_config.name, loaded_config.seeds


## 6. Запуск исследования

Здесь выполняется уже весь перебор по конфигу исследования.


In [ ]:
study_result = study.run_study(loaded_config)
len(study_result['records']), len(study_result['summary'])


In [ ]:
show_table(study_result['summary'])


## 7. График по результатам

Так как summary уже готов, дальше можно строить графики обычными средствами.


In [ ]:
study.plot_study_metric(
    study_result,
    x='matrix.rank',
    y='avg_test_rmse',
    hue='method',
    title='RMSE в зависимости от истинного ранга',
    xlabel='Истинный ранг',
    ylabel='Средний test RMSE',
);


## 8. Сохранение результатов исследования

Эта функция сохраняет:
- сам конфиг исследования;
- manifest сценариев;
- raw records;
- summary.


In [ ]:
output_paths = study.save_study_outputs(study_result, PROJECT_DIR / 'study_outputs' / study_config.name)
output_paths


## 9. Идея дальнейшей работы

Дальше можно заводить отдельные конфиги исследования для:
- ранга;
- шума;
- типа пропусков;
- когерентности;
- сравнения обычного и compact RGD.

То есть исследование теперь хранится как файл, а не только как состояние notebook.
